# [LAB03S03] Análise Descritiva das Métricas DORA (RQ01 a RQ04) e Classificação de Referência (C1)

**Disciplina:** Laboratório de Experimentação de Software  
**Sprint:** S03 | **Issue:** #39  
**Objetivo:** Realizar a análise descritiva das quatro métricas DORA a partir do dataset consolidado da S02, calculando mediana e IQR em todas as variantes obrigatórias, tratando os casos de censura e gerando a classificação DORA de referência (C1).

---

## 1. Importação de Bibliotecas e Configuração do Ambiente

In [ ]:
import sys
from pathlib import Path

# Assegura resolução correta dos módulos do projeto
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent.parent
elif repo_root.name == 'lab03':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from lab03.src.analise_rq01_rq04 import (
    verificar_integridade_dataset,
    classificar_dora_c1_df,
    analisar_rq01_df,
    analisar_rq02_lt,
    analisar_rq03_cfr,
    analisar_rq04_rt,
    analisar_c1_geral,
    gerar_graficos_analise,
    exportar_tabelas_estatisticas,
    exportar_dataset_c1,
    DORA_CATEGORIES,
    DORA_PALETTE,
)

sns.set_theme(style='whitegrid', font='sans-serif')
print('Ambiente configurado com sucesso!')

## 2. Carregamento do Dataset da S02 e Verificação de Integridade

Carrega o CSV final consolidado da S02 (`lab03/data/metricas_dora.csv`) e verifica a conformidade com o dicionário de dados.

In [ ]:
csv_path = repo_root / 'lab03' / 'data' / 'metricas_dora.csv'
df_raw = pd.read_csv(csv_path)

integridade = verificar_integridade_dataset(df_raw)
print(f'Total de repositórios: {integridade["n_total"]}')
print(f'Dataset íntegro e válido? {integridade["valido"]}')
if integridade['avisos']:
    print(f'Avisos: {integridade["avisos"]}')

df_raw.head(3)

## 3. Enquadramento DORA e Classificação de Referência (C1)

Aplica as regras formais da Tabela de Referência (Seção 5 do enunciado) e calcula a pontuação C1 (Elite=4, High=3, Medium=2, Low=1; mediana dos 4 scores arredondada para baixo).

In [ ]:
df_c1 = classificar_dora_c1_df(df_raw)
print('Colunas geradas:', [c for c in df_c1.columns if c.startswith('c1_') or c.endswith('_category')])
df_c1[['full_name', 'c1_df_category', 'c1_lt_category', 'c1_cfr_category', 'c1_rt_category', 'c1_overall_score', 'c1_overall_category']].head(5)

## 4. RQ 01: Deployment Frequency (Releases por Semana)

- **Métrica:** $\text{releases válidas} \div 52.14$
- **Estatísticas Obrigatórias:** Mediana e IQR.

In [ ]:
rq01_res = analisar_rq01_df(df_c1)
pd.DataFrame([rq01_res['estatisticas']])

In [ ]:
print('Distribuição Categórica RQ01:')
for cat in DORA_CATEGORIES:
    print(f'  {cat:6s}: {rq01_res["distribuicao_contagem"][cat]} repos ({rq01_res["distribuicao_percentual"][cat]}%)')

## 5. RQ 02: Lead Time for Changes (Variantes Obrigatórias a e b)

- **Variante (a) Por Release:** Mediana entre releases de $(\text{data release} - \text{commit mais antigo})$.
- **Variante (b) Por Commit:** Mediana de todos os commits de todas as releases $(\text{data release} - \text{commit.author.date})$.
- **Divergência:** Commits antigos isolados elevam a Variante (a), mantendo a Variante (b) mais robusta.

In [ ]:
rq02_res = analisar_rq02_lt(df_c1)
pd.DataFrame([rq02_res['variante_a'], rq02_res['variante_b'], rq02_res['diferenca']])

## 6. RQ 03: Change Failure Rate (Variantes Obrigatórias a e b)

- **Variante (a) Proxy de CI:** Falhas no default branch com `event=push` sobre total de runs válidos.
- **Variante (b) Proxy de Entrega:** Releases seguidas por release corretiva em até 7 dias.
- **Censura:** Releases nos últimos 7 dias da janela ficam fora do denominador.

In [ ]:
rq03_res = analisar_rq03_cfr(df_c1)
print('Censura da Variante b:', rq03_res['censura_entrega'])
pd.DataFrame([rq03_res['variante_a'], rq03_res['variante_b']])

## 7. RQ 04: Failed Deployment Recovery Time (Horas e Censura)

- **Métrica:** Duração dos episódios de falha por workflow até o próximo sucesso.
- **Tratamento de Censura:** Episódios não recuperados até o encerramento da janela são censurados (evitando viés de sobrevivência).

In [ ]:
rq04_res = analisar_rq04_rt(df_c1)
print('Censura Agregada:', rq04_res['censura_agregada'])
pd.DataFrame([rq04_res['tempo_recuperacao'], rq04_res['taxa_censura_repositorio']])

## 8. Classificação DORA de Referência Geral (C1)

Distribuição geral consolidada e comparativo cruzado entre as 4 métricas.

In [ ]:
c1_res = analisar_c1_geral(df_c1)
print('Distribuição Geral DORA C1:')
for cat in DORA_CATEGORIES:
    print(f'  {cat:6s}: {c1_res["distribuicao_contagem"][cat]} repositórios ({c1_res["distribuicao_percentual"][cat]}%)')

c1_res['tabela_comparativa']

## 9. Exportação de Tabelas, Figuras e Dataset de Referência para RQ07

In [ ]:
data_dir = repo_root / 'lab03' / 'data'
fig_dir = data_dir / 'figuras'

tabelas = exportar_tabelas_estatisticas(rq01_res, rq02_res, rq03_res, rq04_res, c1_res, data_dir)
dataset_c1 = exportar_dataset_c1(df_c1, data_dir / 'classificacao_dora_c1.csv')
figs = gerar_graficos_analise(df_c1, fig_dir)

print(f'Tabelas exportadas com sucesso: {[p.name for p in tabelas.values()]}')
print(f'Dataset C1 salvo em: {dataset_c1.name}')
print(f'Total de figuras geradas: {len(figs)}')
for f in figs:
    print(f' - {f.name}')